# PyTorch DType Tradeoffs & Memory Optimization Notes

## 1. Common PyTorch DTypes

| DType | Bits | Memory per Parameter | Speed | Precision | Typical Usage |
|---|---|---|---|---|---|
| `float64` (`torch.double`) | 64 | 8 bytes | Slow | Very High | Scientific computing |
| `float32` (`torch.float`) | 32 | 4 bytes | Standard | High | Default deep learning training |
| `float16` (`torch.float16`) | 16 | 2 bytes | Very Fast | Medium | Mixed precision training |
| `bfloat16` (`torch.bfloat16`) | 16 | 2 bytes | Very Fast | Better stability than FP16 | Modern LLM training |
| `int8` | 8 | 1 byte | Extremely Fast | Lower | Quantized inference |
| `int4` | 4 | 0.5 byte | Extremely Fast | Lower | QLoRA / compressed LLMs |

---

# 2. Memory vs Precision Tradeoffs
## FP32 (`torch.float32`)

### Infos
- 4 bytes per parameter
- Stable gradients, High numerical precision, Reliable training
- BADs:High VRAM usage, Slower training, Smaller batch sizes
- USEs: Standard model training, Research/debugging, Sensitive numerical computations

```python
x = x.float()
```

## FP16 (`torch.float16`)

### Infos
- 2 bytes per parameter
- Faster GPU computation, Lower memory usage, Larger batch sizes
- BADs: Numerical instability, Overflow/underflow issues, Lower precision
- USEs: Mixed precision training, Transformer training, GPU optimization

```python
x = x.half()
```

## BF16 (`torch.bfloat16`)

### Infos
- 2 bytes per parameter
- Better numerical stability than FP16, Large exponent range, Fast computation
- BADs: Slightly lower mantissa precision, Requires modern hardware support
- USEs: Modern LLM training, Distributed training, Large transformer models

```python
x = x.to(torch.bfloat16)
```

## INT8

### Infos
- 1 byte per parameter
- Very low memory usage, Fast inference, Efficient deployment
- BADs: Accuracy degradation, Quantization complexity, Poor training suitability
- USEs: Production inference, CPU inference, Edge deployment

```python
x = x.to(torch.int8)
```

## INT4

### Infos
- 0.5 bytes per parameter
- Extremely low memory usage, Massive compression, Enables large models on small GPUs
- BADs: Significant approximation errors, Lower accuracy, Difficult optimization
- USEs: QLoRA, Quantized LLM inference, Consumer GPU deployment

```python
# Usually handled through quantization libraries
```

---

# 3. Can We Change DTypes for Everything?

## Yes, but with caveats.

### Safe to convert:
- Model weights
- Activations
- Inputs

### Dangerous to fully reduce precision:
- Softmax
- LayerNorm
- Attention score computation
- Loss calculations

These are often kept in FP32 automatically.

---

# 4. Mixed Precision Training

### Infos
- ~50% lower memory usage
- Faster training, Better GPU utilization, Larger batch sizes
- BADs: Possible instability in FP16, More debugging complexity
- USEs: Modern deep learning training, Transformer models, LLM training

```python
from torch.cuda.amp import autocast
```

---

# 5. Real LLM Precision Strategy

| Component | Precision |
|---|---|
| Model weights | BF16 |
| Activations | BF16 |
| Gradients | Mixed |
| Optimizer states | FP32 |
| LayerNorm | FP32 |
| Attention softmax | FP32 |

---

# 6. Biggest Memory Consumers During Training

Memory usage is NOT only model weights.

## Major VRAM Consumers
1. Activations
2. Gradients
3. Optimizer states
4. Attention cache
5. Parameters

---

# 7. Major Memory Optimization Techniques

## Gradient Checkpointing

### Infos
- Reduces activation memory usage
- Enables training larger models, Significant VRAM savings
- BADs: Slower training, Recomputes activations during backward pass
- USEs: Large transformers, LLM fine-tuning, Memory-constrained training

```python
model.gradient_checkpointing_enable()
```
---

## Mixed Precision Training

### Idea
Use FP16/BF16 where possible.

### Memory Savings
- ~50% reduction

### Tradeoff
- Possible instability in FP16

### Use Case
- Almost all modern training pipelines

---

## Quantization

### Infos
- Massive memory reduction
- Faster inference, Smaller deployment size, Lower hardware requirements
- BADs: Accuracy degradation, Quantization complexity
- USEs: LLM serving, Edge AI, Production inference
```python
model.quantize()
```

## LoRA

### Infos
- Trains very few additional parameters
- Extremely memory efficient, Fast fine-tuning, Cheap adaptation
- BADs: Limited adaptation capacity, Requires adapter management
- USEs: LLM fine-tuning, Domain adaptation, Instruction tuning
```python
from peft import LoraConfig
```

## QLoRA
### Infos
- Uses 4-bit quantized weights
- Very low VRAM usage, Enables large model fine-tuning on consumer GPUs
- BADs: Quantization approximation errors, More complex setup
- USEs: Efficient LLM fine-tuning, Consumer GPU training, Resource-constrained systems
```python
from transformers import BitsAndBytesConfig
```

## ZeRO Optimization
### Infos
- Shards optimizer states and gradients across GPUs
- Massive distributed memory savings, Enables huge model training
- BADs: Increased communication overhead, Distributed complexity
- USEs: Multi-GPU training, Large-scale LLM training, Distributed systems
```python
from deepspeed import initialize
```

## FlashAttention
### Infos
- Reduces attention memory complexity
- Faster attention computation, Better scalability, Lower VRAM usage
- BADs: Hardware compatibility constraints, Additional setup complexity
- USEs: Transformer optimization, Long context models, Efficient attention computation
```python
from flash_attn import flash_attn_func
```

---

# 8. PyTorch Useful APIs

## Convert dtype
```python
x = x.to(torch.float16)
```

---

## Check dtype
```python
print(x.dtype)
```

---

## Convert model precision
```python
model = model.half()
```

---

## Use BF16
```python
model = model.to(torch.bfloat16)
```

---

# 9. Practical Recommendations

| Scenario | Recommended Precision |
|---|---|
| Research/debugging | FP32 |
| Standard training | Mixed precision |
| Modern LLM training | BF16 |
| Consumer GPU fine-tuning | QLoRA |
| Production inference | INT8 |
| Extreme compression | INT4 |

---

# 10. Important Industry Insight

Modern AI systems rarely train entirely in FP32 anymore.

Current standard:
- BF16/FP16 computation
- FP32 stability layers
- Quantized inference
- LoRA/QLoRA fine-tuning
- FlashAttention optimization
- Distributed ZeRO sharding

This combination enables:
- Faster training
- Lower costs
- Larger models
- Consumer hardware deployment